**[🏠 Course Home](00_START_HERE.ipynb)** | [⬅️ Previous: Sheet 2 (Quadratic Approx)](02_quadratic_laplace_approximation.ipynb) | **Sheet 3 of 4: First Principles** | [➡️ Next: Sheet 4 (Production MCMC)](04_mcmc_production_diagnostics.ipynb)

---

# Sheet 3: The Mechanics of MCMC: How Random Walks Find Truth
### *The Pure First-Principles Bridge (Zero External Libraries)*

This notebook demonstrates how a blind, memoryless random walker explores parameter space and converges to the exact posterior distribution using **Detailed Balance** and the **Metropolis-Hastings algorithm** in pure Base R.

## Part 1: The Core Mechanics (Propose $\to$ Ratio $\to$ Coin Flip)

In [ ]:
set.seed(42)
target_log_prob <- function(theta) {
  dnorm(theta, mean = 172.5, sd = 8.0, log = TRUE)
}

mcmc_single_step <- function(current_theta, step_size = 2.0) {
  proposed_theta <- current_theta + rnorm(1, mean = 0, sd = step_size)
  log_alpha <- target_log_prob(proposed_theta) - target_log_prob(current_theta)
  if (log(runif(1)) < log_alpha) {
    return(list(theta = proposed_theta, accepted = TRUE))
  } else {
    return(list(theta = current_theta,  accepted = FALSE))
  }
}

run_simple_mcmc <- function(n_steps = 3000, step_size = 2.0, start_theta = 150) {
  samples <- numeric(n_steps)
  curr <- start_theta
  accepts <- 0
  for (t in 1:n_steps) {
    res <- mcmc_single_step(curr, step_size = step_size)
    if (res$accepted) accepts <- accepts + 1
    curr <- res$theta
    samples[t] <- curr
  }
  return(list(samples = samples, rate = accepts / n_steps))
}

## Part 2: The Step-Size Failure Modes (Tuning $\tau$)
1. **Too Small ($\tau = 0.2$)**: $98\%$ acceptance, moves like molasses (high autocorrelation).
2. **Too Large ($\tau = 40.0$)**: $1\%$ acceptance, gets frozen on flatlines.
3. **Optimal ($\tau = 3.5$)**: Healthy mixing with $20\% - 50\%$ acceptance rate.

In [ ]:
set.seed(42)
chain_small <- run_simple_mcmc(3000, step_size = 0.2)
chain_large <- run_simple_mcmc(3000, step_size = 40.0)
chain_opt   <- run_simple_mcmc(3000, step_size = 3.5)

par(mfrow = c(3, 1), mar = c(3, 4, 2, 1))
plot(chain_small$samples, type = "l", col = "darkred", las = 1,
     main = sprintf("1. Step Size Too Small (tau = 0.2) | Acceptance = %.1f%% (Slow Diffusion)", chain_small$rate * 100), ylab = "theta")
plot(chain_large$samples, type = "l", col = "orange", las = 1,
     main = sprintf("2. Step Size Too Large (tau = 40.0) | Acceptance = %.1f%% (Frozen Flatlines)", chain_large$rate * 100), ylab = "theta")
plot(chain_opt$samples, type = "l", col = "darkblue", las = 1,
     main = sprintf("3. Optimal Step Size (tau = 3.5) | Acceptance = %.1f%% (Healthy Mixing)", chain_opt$rate * 100), ylab = "theta")
par(mfrow = c(1, 1))

## Part 3: Autocorrelation (`acf()`) & Memory Decay

In [ ]:
par(mfrow = c(1, 2))
acf(chain_small$samples, lag.max = 40, col = "darkred", lwd = 2, main = "Small Step: High Autocorrelation")
acf(chain_opt$samples, lag.max = 40, col = "darkblue", lwd = 2, main = "Optimal Step: Rapid Memory Decay")
par(mfrow = c(1, 1))

## Part 4: Hands-On Challenge Exercises

### Exercise 1: Exploring Thinning
If a chain has high autocorrelation, what happens if you keep only every 5th sample (`chain_small$samples[seq(1, 3000, by = 5)]`)? Plot the `acf()` of the thinned chain.

### Exercise 2: Why Detailed Balance Matters
If we always accepted proposals ($lpha = 1.0$), what distribution would the chain explore? *(Hint: A pure Gaussian random walk exploring uniform infinite space, never settling on the posterior).* 

---

**[🏠 Course Home](00_START_HERE.ipynb)** | [⬅️ Previous: Sheet 2 (Quadratic Approx)](02_quadratic_laplace_approximation.ipynb) | [➡️ Next: Sheet 4 (Production MCMC)](04_mcmc_production_diagnostics.ipynb)